In [1]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, LSTM, Dense, Embedding
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
eng = [
    "hello",
    "good morning",
    "good afternoon",
    "good evening",
    "good night",
    "thank you",
    "you are welcome",
    "how are you",
    "i am fine",
    "i am happy",
    "i am sad",
    "i am tired",
    "i am hungry",
    "i am thirsty",
    "i love you",
    "i miss you",
    "goodbye",
    "see you soon",
    "see you tomorrow",
    "nice to meet you",
    "what is your name",
    "my name is john",
    "where are you",
    "where do you live",
    "i live in paris",
    "i am going home",
    "i like coffee",
    "i like tea",
    "i like music",
    "i like football",
    "i love my family",
    "this is good",
    "this is bad",
    "this is beautiful",
    "i am a student",
    "i am a teacher",
    "she is my friend",
    "he is my brother",
    "she is my sister",
    "i have a book",
    "i have a car",
    "open the door",
    "close the door",
    "come here",
    "please sit down",
    "please help me",
    "what time is it",
    "today is monday",
    "i speak english",
    "i want to learn french"
]

fra = [
    "bonjour",
    "bonjour",
    "bon apres midi",
    "bonsoir",
    "bonne nuit",
    "merci",
    "de rien",
    "comment allez vous",
    "je vais bien",
    "je suis heureux",
    "je suis triste",
    "je suis fatigue",
    "j ai faim",
    "j ai soif",
    "je vous aime",
    "vous me manquez",
    "au revoir",
    "a bientot",
    "a demain",
    "ravi de vous rencontrer",
    "comment vous appelez vous",
    "je m appelle john",
    "ou etes vous",
    "ou habitez vous",
    "j habite a paris",
    "je rentre a la maison",
    "j aime le cafe",
    "j aime le the",
    "j aime la musique",
    "j aime le football",
    "j aime ma famille",
    "c est bon",
    "c est mauvais",
    "c est beau",
    "je suis etudiant",
    "je suis professeur",
    "elle est mon amie",
    "il est mon frere",
    "elle est ma soeur",
    "j ai un livre",
    "j ai une voiture",
    "ouvrez la porte",
    "fermez la porte",
    "venez ici",
    "asseyez vous s il vous plait",
    "aidez moi s il vous plait",
    "quelle heure est il",
    "aujourd hui c est lundi",
    "je parle anglais",
    "je veux apprendre le francais"
]

print("Number of English sentences:", len(eng))
print("Number of French sentences:", len(fra))

Number of English sentences: 50
Number of French sentences: 50


In [3]:
fra_input = ["<start> " + x for x in fra]
fra_target = [x + " <end>" for x in fra]

eng_tokenizer = Tokenizer(filters="")
fra_tokenizer = Tokenizer(filters="")

eng_tokenizer.fit_on_texts(eng)
fra_tokenizer.fit_on_texts(fra_input + fra_target)

X = pad_sequences(
    eng_tokenizer.texts_to_sequences(eng),
    padding="post"
)

Y = pad_sequences(
    fra_tokenizer.texts_to_sequences(fra_input),
    padding="post"
)

T = pad_sequences(
    fra_tokenizer.texts_to_sequences(fra_target),
    padding="post"
)

T = np.expand_dims(T, -1)

eng_vocab = len(eng_tokenizer.word_index) + 1
fra_vocab = len(fra_tokenizer.word_index) + 1

eng_len = X.shape[1]
fra_len = Y.shape[1]

print(X.shape)
print(Y.shape)
print(T.shape)

(50, 5)
(50, 7)
(50, 7, 1)


In [4]:
encoder_input = Input(shape=(eng_len,))

encoder_embedding = Embedding(
    eng_vocab,
    128
)(encoder_input)

encoder_lstm = LSTM(
    128,
    return_state=True
)

encoder_output, state_h, state_c = encoder_lstm(
    encoder_embedding
)

print("Encoder created")

Encoder created


In [5]:
decoder_input = Input(shape=(fra_len,))

decoder_embedding = Embedding(
    fra_vocab,
    128
)(decoder_input)

decoder_lstm = LSTM(
    128,
    return_sequences=True
)

decoder_output = decoder_lstm(
    decoder_embedding,
    initial_state=[state_h, state_c]
)

output = Dense(
    fra_vocab,
    activation="softmax"
)(decoder_output)

print("Decoder created")

Decoder created


In [6]:
model = Model(
    [encoder_input, decoder_input],
    output
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 5)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_1       │ (None, 7)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, 5, 128)    │     10,496 │ input_layer[0][0] │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_1         │ (None, 7, 128)    │     11,776 │ input_layer_1[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ [(None, 128),     │    131,584 │ embedding[0][0]   │
│                     │ (None, 128),      │            │                   │
│                     │ (None, 128)]      │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 7, 128)    │    131,584 │ embedding_1[0][0… │
│                     │                   │            │ lstm[0][1],       │
│                     │                   │            │ lstm[0][2]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 7, 92)     │     11,868 │ lstm_1[0][0]      │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 297,308 (1.13 MB)

 Trainable params: 297,308 (1.13 MB)

 Non-trainable params: 0 (0.00 B)

In [7]:
history_50 = model.fit(
    [X, Y],
    T,
    epochs=50,
    batch_size=2,
    verbose=1
)

loss_50, accuracy_50 = model.evaluate(
    [X, Y],
    T,
    verbose=0
)

print("Epochs = 50")
print("Loss:", loss_50)
print("Accuracy:", accuracy_50)

Epoch 1/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step - accuracy: 0.3800 - loss: 3.8644     
Epoch 2/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4571 - loss: 2.5363 
Epoch 3/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.4800 - loss: 2.3337 
Epoch 4/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.5343 - loss: 2.2043 
Epoch 5/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5571 - loss: 2.1203 
Epoch 6/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5629 - loss: 2.0493 
Epoch 7/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5686 - loss: 1.9736 
Epoch 8/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.5629 - loss: 1.8834 
Epoch 9/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6000 - loss: 1.8082 
Epoch 10/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5829 - loss: 1.7344 
Epoch 11/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.6229 - loss: 1.6058 
Epoch 12/50
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accura

In [8]:
history_100 = model.fit(
    [X, Y],
    T,
    epochs=100,
    batch_size=2,
    verbose=1
)

loss_100, accuracy_100 = model.evaluate(
    [X, Y],
    T,
    verbose=0
)

print("Epochs = 100")
print("Loss:", loss_100)
print("Accuracy:", accuracy_100)

Epoch 1/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9800 - loss: 0.0872 
Epoch 2/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9771 - loss: 0.0786 
Epoch 3/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9743 - loss: 0.0767 
Epoch 4/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9829 - loss: 0.0669 
Epoch 5/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9829 - loss: 0.0610 
Epoch 6/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9829 - loss: 0.0546 
Epoch 7/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9914 - loss: 0.0484 
Epoch 8/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9886 - loss: 0.0474 
Epoch 9/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9943 - loss: 0.0450 
Epoch 10/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9914 - loss: 0.0457 
Epoch 11/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.9943 - loss: 0.0411 
Epoch 12/100
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 

In [9]:
history_150 = model.fit(
    [X, Y],
    T,
    epochs=150,
    batch_size=2,
    verbose=1
)

loss_150, accuracy_150 = model.evaluate(
    [X, Y],
    T,
    verbose=0
)

print("Epochs = 150")
print("Loss:", loss_150)
print("Accuracy:", accuracy_150)

Epoch 1/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0023 
Epoch 2/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0022 
Epoch 3/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0022 
Epoch 4/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0022 
Epoch 5/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0021 
Epoch 6/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 1.0000 - loss: 0.0021 
Epoch 7/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0020 
Epoch 8/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0020 
Epoch 9/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0020 
Epoch 10/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 1.0000 - loss: 0.0019 
Epoch 11/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 1.0000 - loss: 0.0019 
Epoch 12/150
25/25 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 

In [10]:
print("==============================")
print("Epoch Comparison")
print("==============================")

print("50 Epochs  :", accuracy_50)
print("100 Epochs :", accuracy_100)
print("150 Epochs :", accuracy_150)

Epoch Comparison
50 Epochs  : 0.9885714650154114
100 Epochs : 1.0
150 Epochs : 1.0


(4) What are state_h and state_c?

state_h is the hidden state, and state_c is the cell state. They carry information from the encoder to the decoder.

(5) What is Teacher Forcing?

Teacher forcing is a training method where the decoder uses the actual previous target word as input instead of its own previous prediction.

(6) What happens when the sentence becomes very long?

When the sentence becomes very long, an LSTM may have difficulty remembering information from the beginning. This can reduce translation accuracy and cause information loss.